In [9]:
SAMPLE_JOB = """高级 Python 工程师

我们正在寻找一位高级 Python 工程师，加入我们的 API 平台团队。

职位要求：
- 5 年以上 Python 开发经验
- 具备分布式系统开发经验
- 深入理解 REST API 和微服务架构
- 熟悉 PostgreSQL、Redis
- 有 Kubernetes 使用经验者优先
- 具备良好的沟通能力

岗位职责：
- 设计并开发高性能 API，每天处理数百万次请求
- 主导技术设计评审
- 指导和培养初级工程师
- 与产品经理合作，评估产品需求的技术可行性
"""

In [10]:
SAMPLE_CANDIDATE = """
Jane Doe —— 7 年 Python 开发经验

当前职位：DataCorp 高级软件工程师

技能：
- Python
- FastAPI
- Django
- PostgreSQL
- Redis
- Docker
- Kubernetes
- AWS

主要成就：
- 构建了每天处理 500 万次请求的 API 平台
- 带领 4 人工程团队
- 将 API 延迟降低了 40%
- 指导培养了 3 名初级工程师

教育背景：
加州大学伯克利分校（UC Berkeley）计算机科学学士
"""

In [3]:
ANALYST_TASK_PROMPT ="""
请分析以下职位描述：

{job_desc}

请提取并分析：
1. 最重要的 5 项技能要求
2. 企业文化信号（Culture Signals）
3. 公司最看重的能力与价值观
4. 可能存在的风险点（Potential Red Flags）
5. 求职材料中应重点呼应和使用的关键词或表达方式
"""

In [11]:
APPLICATION_TASK_PROMPT ="""
请基于职位分析结果，为以下候选人制作定制化求职材料：

{candidate_profile}

请输出以下内容：

1. **求职信（Cover Letter）**
   - 250～300 字
   - 共 3 个自然段：
     - 第一段：吸引招聘者的开场（Hook）
     - 第二段：展示与岗位高度匹配的能力和经历（Evidence）
     - 第三段：总结并表达加入公司的意愿（Close）

2. **简历中最值得突出的 5 条经历（Resume Bullets）**
   - 针对该岗位进行优化
   - 突出与岗位最相关的成果和能力

3. **10 个最可能出现的面试问题**
   - 5 个行为面试题（Behavioral）
   - 5 个技术面试题（Technical）
   - 为每个问题提供建议的回答思路（Answer Framework）

4. **薪资谈判区间建议**
   - 根据岗位级别和公司情况，估算合理的薪资谈判范围
"""

In [12]:
from dotenv import load_dotenv

load_dotenv()

import os

model = os.getenv('MODEL_NAME', '')
api_base = os.getenv('OPENAI_API_BASE', '')

from crewai import Agent, Crew, Task, LLM, Process

import asyncio
import nest_asyncio

nest_asyncio.apply()

def run_job_application_crew(job_desc: str, candidate_profile: str):
    llm = LLM(model=model, temperature=0.2, api_base=api_base)

    analyst = Agent(
        role="职位需求分析师",
        goal="分析职位描述，识别关键技能要求、企业价值观以及企业文化信号",
        backstory="曾任大厂招聘经理，拥有 10 年招聘经验，擅长深入解读职位描述，精准挖掘企业真正关注的人才特质。",
        llm=llm,
        verbose=True,
    )

    writer = Agent(
        role="职业顾问与求职材料撰写专家",
        goal="为候选人量身定制求职材料，最大限度提升获得面试机会的概率",
        backstory="资深职业顾问，曾帮助 500 多位候选人成功入职全球顶尖科技公司。",
        llm=llm,
        verbose=True,
    )

    analyst_task = Task(
        description=ANALYST_TASK_PROMPT.format(job_desc=job_desc),
        expected_output="职位需求分析报告，包括关键技能要求、企业价值观、企业文化信号、潜在风险点及重要关键词等。",
        agent=analyst,
    )

    application_task = Task(
        description=APPLICATION_TASK_PROMPT.format(candidate_profile=candidate_profile),
        expected_output="包含求职信、简历优化建议、面试问题及薪资区间的完整求职材料。",
        agent=writer,
        context=[analyst_task],
    )

    crew = Crew(
        agents=[analyst, writer],
        tasks=[analyst_task, application_task],
        process=Process.sequential,
        verbose=False,
    )

    result = asyncio.run(crew.kickoff_async())
    return str(result)


In [13]:
response = run_job_application_crew(job_desc=SAMPLE_JOB, candidate_profile=SAMPLE_CANDIDATE)

print(response)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 职位需求分析师                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│  请分析以下职位描述：                                                                                           │
│                                                                                                                 │
│  高级 Python 工程师                                                                                             │
│                                                                                                                 │
│  我们正在寻找一位高级 Python 工程师，加入我们的 API 平台团队。                                                  │
│                                                                                                                 │
│  职位要求：                                                                                                     │
│  - 5 年以上 Python 开发经验                                                                                     │
│  - 具备分布式系统开发经验                                                                                       │
│  - 深入理解 REST API 和微服务架构                                                                               │
│  - 熟悉 PostgreSQL、Redis                                                                                       │
│  - 有 Kubernetes 使用经验者优先                                                                                 │
│  - 具备良好的沟通能力                                                                                           │
│                                                                                                                 │
│  岗位职责：                                                                                                     │
│  - 设计并开发高性能 API，每天处理数百万次请求                                                                   │
│  - 主导技术设计评审                                                                                             │
│  - 指导和培养初级工程师                                                                                         │
│  - 与产品经理合作，评估产品需求的技术可行性                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│  请提取并分析：                                                                                                 │
│  1. 最重要的 5 项技能要求                                                                                       │
│  2. 企业文化信号（Culture Signals）                                                                             │
│  3. 公司最看重的能力与价值观                                                                                    │
│  4. 可能存在的风险点（Potential Red Flags）                                                                     │
│  5. 求职材料中应重点呼应和使用的关键词或表达方式                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 职位需求分析师                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 职位需求分析报告：高级 Python 工程师（API 平台团队）                                                         │
│                                                                                                                 │
│  基于 10                                                                                                        │
│  年大厂招聘与技术团队管理经验，结合该职位描述（JD）的显性要求与隐性信号，以下为深度拆解报告。本报告旨在帮助候   │
│  选人精准对齐企业预期，识别潜在风险，并优化求职材料策略。                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. 最重要的 5 项技能要求（按业务影响权重排序）                                                              │
│                                                                                                                 │
│  | 排名 | 技能维度 | JD 依据与深度解读 |                                                                        │
│  |:---:|:---|:---|                                                                                              │
│  | **1** | **高并发/高性能 API 架构与开发能力** | 职责明确“每天处理数百万次请求”，这意味着候选人不能仅停留在    │
│  CRUD                                                                                                           │
│  层面，必须具备异步编程、连接池管理、限流熔断、负载均衡、延迟优化等实战经验。框架偏好虽未写明，但实际极可能指   │
│  向 FastAPI/aiohttp 或高度定制化的 Django/Flask。 |                                                             │
│  | **2** | **分布式系统与微服务治理经验** | 明确要求“分布式系统开发经验”与“微服务架构”。API                     │
│  平台团队通常承担服务路由、网关、服务发现、配置中心、链路追踪等基础设施职能，候选人需理解服务拆分边界、分布式   │
│  事务、一致性协议及故障隔离策略。 |                                                                             │
│  | **3** | **数据存储与缓存架构优化（PostgreSQL + Redis）** |                                                   │
│  两者并列出现，说明数据层是性能瓶颈的核心战场。需掌握 PG 的索引优化、查询计划分析、分区表/分库分表策略，以及    │
│  Redis 的缓存穿透/击穿/雪崩防护、持久化策略、集群模式与热点 Key 治理。 |                                        │
│  | **4** | **技术领导力与架构评审能力** |                                                                       │
│  “主导技术设计评审”与“指导培养初级工程师”表明该岗位并非纯执行岗，而是团队的技术锚点。需具备撰写                 │
│  RFC/ADR（架构决策记录）、组织跨团队评审、把控技术选型风险、建立代码规范与 Review 机制的能力。 |                │
│  | **5** | **跨职能协作与业务技术转化能力** |                                                                   │
│  “与产品经理合作评估技术可行性”+“良好沟通能力”被单独强调，说明团队处于业务快速迭代期，工程师需具备将模糊需求转  │
│  化为可落地技术方案的能力，并能用非技术语言对齐预期、管理需求边界。 |                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. 企业文化信号（Culture Signals）                                                                          │
│                                                                                                                 │
│  | JD 原文片段 | 文化信号解读 | 团队/组织特征推断 |                                                             │
│  |:---|:---|:---|                                                                                               │
│  | `每天处理数百万次请求` | **工程

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 职业顾问与求职材料撰写专家                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│  请基于职位分析结果，为以下候选人制作定制化求职材料：                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  Jane Doe —— 7 年 Python 开发经验                                                                               │
│                                                                                                                 │
│  当前职位：DataCorp 高级软件工程师                                                                              │
│                                                                                                                 │
│  技能：                                                                                                         │
│  - Python                                                                                                       │
│  - FastAPI                                                                                                      │
│  - Django                                                                                                       │
│  - PostgreSQL                                                                                                   │
│  - Redis                                                                                                        │
│  - Docker                                                                                                       │
│  - Kubernetes                                                                                                   │
│  - AWS                                                                                                          │
│                                                                                                                 │
│  主要成就：                                                                                                     │
│  - 构建了每天处理 500 万次请求的 API 平台                                                                       │
│  - 带领 4 人工程团队                                                                                            │
│  - 将 API 延迟降低了 40%                                                                                        │
│  - 指导培养了 3 名初级工程师                                                                                    │
│                                                                                                                 │
│  教育背景：                                                                                                     │
│  加州大学伯克利分校（UC Berkeley）计算机科学学士                                                                │
│                                                                                                                 │
│                                                                                                                 │
│  请输出以下内容：                                                                                               │
│                                                                                                                 │
│  1. **求职信（Cover Letter）**                                                                                  │
│     - 250～300 字                                                              

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 职业顾问与求职材料撰写专家                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 定制化求职材料包（Jane Doe｜高级 Python 工程师）                                                             │
│                                                                                                                 │
│  ## 1. 求职信（Cover Letter）                                                                                   │
│  尊敬的招聘负责人：您好！在 DataCorp 担任高级软件工程师的七年里，我专注于构建高可用 API                         │
│  基础设施。贵司对极致工程标准与业务赋能的追求与我高度契合，特此申请高级 Python 工程师一职。                     │
│                                                                                                                 │
│  我曾主导日均 500 万次请求的 API 平台架构。通过 FastAPI 异步重构、PostgreSQL 索引优化与 Redis                   │
│  多级缓存，将核心接口 P99 延迟降低 40%，并基于 Kubernetes 实现弹性部署。管理上，我带领 4                        │
│  人小组主导技术评审，建立 Code Review 与导师机制，辅导 3 名初级工程师晋升，有效降低线上故障率。                 │
│                                                                                                                 │
│  我期待将高并发治理与团队技术赋能经验带入贵司，与产研团队紧密协作，打造支撑业务增长的 API                       │
│  基石。感谢审阅，期待进一步交流。                                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. 简历中最值得突出的 5 条经历（Resume Bullets）                                                            │
│  1. **高并发 API 平台架构与性能调优**：主导设计日均 500 万次请求的核心 API 网关，基于 FastAPI + asyncio         │
│  重构同步阻塞链路，结合 PostgreSQL 慢查询治理与 Redis 热点 Key 防护策略，将核心接口 P99 延迟降低                │
│  40%，系统吞吐量提升 2.5 倍。                                                                                   │
│  2. **云原生部署与微服务治理**：基于 Docker 与 Kubernetes 搭建容器化部署流水线，配置 HPA 弹性伸缩与 Service     │
│  Mesh 流量治理；落地全链路可观测性（Metrics/Logs/Traces），保障 API 平台 99.95% SLA 稳定运行。                  │
│  3. **技术评审主导与架构决策**：牵头 10+ 次跨团队技术设计评审，输出 RFC/ADR                                     │
│  文档规范技术选型；通过限流熔断降级机制与数据库分库分表预案，有效隔离分布式系统故障，降低跨服务调用雪崩风险。   │
│  4. **团队建设与导师制落地**：带领 4 人工程小组，建立标准化 Code Review 与自动化测试流程；实施一对一            │
│  Mentorship，辅导 3 名初级工程师掌握高并发编程与性能调优，1 年内全部晋升至中级，团队交付效率提升 30%。          │
│  5. **产研协同与技术可行性评估**：前置参与产品需求评审，通过技术成本建模与架构可行性分析，推动 3                │
│  个高复杂度需求拆分迭代；以非技术语言对齐业务预期，研发交付周期缩短 25%，有效管控技术债。                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 3. 10 个最可能出现的面试问题及回答思路                                                                      │
│                                                                                                                 │
│  ### 🔹 行为面试题（Behavioral）                                                                                │
│  | 问题 | 回答框架（Answer Framework） |                                                                        │
│  |:---|:---|                                                                                                    │
│  | **1. 请分享一次你主导技术设计评审并推动团队达成共识的经历。** | **背景**：复杂需求

# 定制化求职材料包（Jane Doe｜高级 Python 工程师）

## 1. 求职信（Cover Letter）
尊敬的招聘负责人：您好！在 DataCorp 担任高级软件工程师的七年里，我专注于构建高可用 API 基础设施。贵司对极致工程标准与业务赋能的追求与我高度契合，特此申请高级 Python 工程师一职。

我曾主导日均 500 万次请求的 API 平台架构。通过 FastAPI 异步重构、PostgreSQL 索引优化与 Redis 多级缓存，将核心接口 P99 延迟降低 40%，并基于 Kubernetes 实现弹性部署。管理上，我带领 4 人小组主导技术评审，建立 Code Review 与导师机制，辅导 3 名初级工程师晋升，有效降低线上故障率。

我期待将高并发治理与团队技术赋能经验带入贵司，与产研团队紧密协作，打造支撑业务增长的 API 基石。感谢审阅，期待进一步交流。

---

## 2. 简历中最值得突出的 5 条经历（Resume Bullets）
1. **高并发 API 平台架构与性能调优**：主导设计日均 500 万次请求的核心 API 网关，基于 FastAPI + asyncio 重构同步阻塞链路，结合 PostgreSQL 慢查询治理与 Redis 热点 Key 防护策略，将核心接口 P99 延迟降低 40%，系统吞吐量提升 2.5 倍。
2. **云原生部署与微服务治理**：基于 Docker 与 Kubernetes 搭建容器化部署流水线，配置 HPA 弹性伸缩与 Service Mesh 流量治理；落地全链路可观测性（Metrics/Logs/Traces），保障 API 平台 99.95% SLA 稳定运行。
3. **技术评审主导与架构决策**：牵头 10+ 次跨团队技术设计评审，输出 RFC/ADR 文档规范技术选型；通过限流熔断降级机制与数据库分库分表预案，有效隔离分布式系统故障，降低跨服务调用雪崩风险。
4. **团队建设与导师制落地**：带领 4 人工程小组，建立标准化 Code Review 与自动化测试流程；实施一对一 Mentorship，辅导 3 名初级工程师掌握高并发编程与性能调优，1 年内全部晋升至中级，团队交付效率提升 30%。
5. **产研协同与技术可行性评估**：前置参与产品需求评审